# Entrepinares · Cuaderno 1: fundamentos (sin conexión)

**Qué vas a aprender**
1. Qué datos usa la demo (un inventario **ficticio** de activos IT/OT).
2. Qué son las *herramientas* de un agente y cómo se ejecutan **sin** modelo de lenguaje.
3. Cuál es el *contrato* de decisión del Orquestador.
4. Cómo está construido el workflow de **tres agentes** y cuándo se llama a cada uno.
5. Cómo viaja la decisión de un agente al siguiente.

**Condiciones de este cuaderno**
- 🟢 **No usa red, ni Azure, ni Foundry, ni credenciales.** No consume nada.
- 🟢 **No modifica ningún dato**: al final se comprueba con SHA-256 que `datos/` sigue idéntico.
- Duración aproximada: 10 minutos. Ejecuta las celdas **en orden** (`Shift + Enter`).

> Los datos son ficticios: no hay escaneo real de equipos ni conexión con sistemas de ninguna empresa.

## 1. Preparación segura

La celda siguiente:
- localiza la raíz del repositorio y carga el pequeño adaptador `entrepinares_utils.py`;
- guarda la **huella SHA-256** de todos los archivos de `datos/` (para demostrar al final que no cambian);
- **bloquea las descargas de CISA** *antes* de usar ninguna herramienta.

¿Por qué bloquear? Las herramientas consultan el catálogo CISA KEV. Si la copia local tiene más de 24 h, el código original intentaría **descargarlo y sobrescribir** `datos/cisa_kev_cache.json`. En esta formación queremos consultas de solo lectura, así que se desactiva la descarga en este proceso.

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repositorio (funciona abriendo el cuaderno desde notebooks/ o desde la raíz)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos" / "inventario_demo.csv").exists())
sys.path.insert(0, str(RAIZ / "notebooks"))
import entrepinares_utils as u

huellas_antes = u.huellas_datos()
m = u.preparar_entorno(cargar_env=False)   # offline: no necesitamos .env ni endpoint

import fuente_kev
try:
    fuente_kev.descargar_catalogo()
except RuntimeError as e:
    print("Descarga de CISA bloqueada ✔  ->", e)
print("Archivos de datos protegidos:", len(huellas_antes))
print("Módulo del workflow cargado :", m.__name__)
print("Modelo configurado          :", m.MODEL)

## 2. El inventario ficticio

Once activos de una empresa imaginaria: servidores de impresión, firewalls y equipos industriales (OT). Fíjate en `fabricante`, `producto` y `version`: son los datos que se cruzan con el catálogo CISA KEV.

In [ ]:
from inventario import leer_inventario

inventario = leer_inventario()
columnas = ["asset_id", "nombre", "tipo", "fabricante", "producto", "version", "criticidad"]
anchos = [max(len(str(a[c])) for a in inventario + [dict(zip(columnas, columnas))]) for c in columnas]
print("  ".join(c.ljust(w) for c, w in zip(columnas, anchos)))
print("  ".join("-" * w for w in anchos))
for a in inventario:
    print("  ".join(str(a[c]).ljust(w) for c, w in zip(columnas, anchos)))
print(f"\n{len(inventario)} activos")

## 3. Las herramientas: las «manos» de los agentes

Un agente con LLM **decide** qué herramienta llamar; la herramienta en sí es **código Python normal y determinista**. Aquí las llamamos directamente con `.func(...)` (sin modelo).

> Las líneas que empiezan por `>>>` las imprime la propia herramienta al ejecutarse.

### 3.1 `consultar_activo`: lectura de un activo del inventario

In [ ]:
print(u.json_bonito(m.consultar_activo.func("IT-004")))
print()
print("Un activo que NO existe:")
print(u.json_bonito(m.consultar_activo.func("IT-999")))

### 3.2 `consultar_cribado`: coincidencias con CISA KEV

El *cribado* compara **fabricante y producto** del activo con el catálogo CISA. Fíjate en dos detalles:
- **(A)** son las coincidencias candidatas del producto del activo; **(B)** son de *otro producto del mismo fabricante* y **no se suman** a (A).
- Son **candidatas, no vulnerabilidades confirmadas**: el cribado no comprueba la versión instalada.

In [ ]:
import json

cribado = json.loads(m.consultar_cribado.func("IT-004", ""))
A = cribado["A_coincidencias_candidatas_producto_o_familia"]
B = cribado["B_coincidencias_solo_fabricante_otro_producto"]
print("Activo       :", cribado["activo_inventario"])
print("Método       :", cribado["metodo"])
print(f"(A) candidatas del producto : {A['total']}  | desglose: {A['desglose_por_producto_cisa']}")
print(f"(B) solo fabricante (otro)  : {B['total']}  | desglose: {B['desglose_por_producto_cisa']}")
print("Advertencia  :", cribado["advertencia"])
print("\nPrimeras 3 candidatas de (A):")
for fila in A["cves_mostradas"][:3]:
    print("  ", fila)

### 3.3 `evaluar_activo`: reglas deterministas de afectación

Nuestro motor solo tiene reglas específicas para **tres CVE**. Aquí evaluamos el firewall `IT-006`, cuya versión incluye un *hotfix*. Observa el `estado` y la `prioridad`.

In [ ]:
evaluacion = json.loads(m.evaluar_activo.func("IT-006"))
print("Activo:", evaluacion["activo"]["asset_id"], "-", evaluacion["activo"]["producto"], evaluacion["activo"]["version"])
for h in evaluacion["hallazgos"]:
    print(f"\n{h['cve']}: estado={h['estado']} | prioridad={h['prioridad']} | en KEV={h['en_kev']}")
    print("  Justificación:", h["justificacion"])

### 3.4 `consultar_cisa`: estado del catálogo

Muestra la versión del catálogo guardado y su antigüedad. Verás un aviso: la copia local es **antigua** y la actualización está **bloqueada** (lo hicimos nosotros en el paso 1). Es el comportamiento deseado en esta formación.

In [ ]:
cisa = json.loads(m.consultar_cisa_solo_lectura.func())
for clave in ("modo", "version_catalogo", "total_cves", "ultima_descarga", "aviso"):
    print(f"{clave:17}: {cisa.get(clave)}")

## 4. El contrato de decisión del Orquestador

El primer agente **no responde con texto libre**: devuelve un objeto con una estructura fija (`DecisionOrquestador`, un modelo *Pydantic*). Así el workflow decide la ruta con datos tipados y no leyendo frases.

In [ ]:
from pydantic import ValidationError

esquema = m.DecisionOrquestador.model_json_schema()
for campo, info in esquema["properties"].items():
    print(f"- {campo:17} {info.get('enum') or info.get('description', '')[:70]}")

ejemplo = m.DecisionOrquestador(
    ruta="analisis_avanzado", asset_id="IT-004", cve_id=None,
    accion_explicita="ninguna", respuesta_directa=None,
    motivo="Pide análisis y prioridad de un activo")
print("\nEjemplo válido:\n", ejemplo.model_dump_json(indent=2))

try:
    m.DecisionOrquestador(ruta="inventada", asset_id=None, cve_id=None,
                          accion_explicita="ninguna", respuesta_directa=None, motivo="x")
except ValidationError as e:
    print("\nUna ruta no permitida se rechaza ✔ ->", e.errors()[0]["msg"])

## 5. Los tres agentes y sus herramientas

Construimos el workflow **sin conectarnos a nada**: el cliente de Foundry se crea con un endpoint ficticio (`ejemplo.invalid`) y *nunca se usa*. Sirve para inspeccionar la estructura.

Fíjate en que **ningún agente tiene herramientas de escritura**: en este paquete el workflow se construye siempre en modo solo lectura.

In [ ]:
workflow, agentes = u.construir_workflow_sin_conexion(m)

for nombre, agente in agentes.items():
    herramientas = [t.name for t in agente.default_options.get("tools", [])]
    print(f"{nombre}\n   {agente.description}\n   herramientas: {herramientas}\n")

todas = {t.name for a in agentes.values() for t in a.default_options.get("tools", [])}
escritura = {"actualizar_novedades_cisa_controlada", "generar_informes_controlados"}
print("¿Alguna herramienta de escritura registrada?", bool(todas & escritura))

## 6. El grafo: cuatro nodos y aristas condicionales

Las líneas discontinuas (`conditional`) significan que el workflow **decide en tiempo de ejecución** si sigue por ese camino. Resumen de las reglas del código:

| Desde | Va a… | Cuando… |
|---|---|---|
| Orquestador | **RespuestaFinal** | `ruta = respuesta_directa`, el activo no existe, o se pide una acción de escritura (que en solo lectura no se ejecuta) |
| Orquestador | **AgenteCorrelacion** | `ruta` es `correlacion` o `analisis_avanzado`, sin acción de escritura y con activo existente |
| AgenteCorrelacion | **AgenteCriticidad** | la ruta era `analisis_avanzado` |
| AgenteCorrelacion | **RespuestaFinal** | en los demás casos |
| AgenteCriticidad | **RespuestaFinal** | siempre |

Por eso una pregunta sencilla (*«¿qué versión tiene IT-004?»*) **no llama** a los agentes especialistas.

In [ ]:
u.mostrar_mermaid(u.diagrama_mermaid(workflow))

> **Nota:** si no ves el dibujo, tu visor no renderiza *Mermaid* (JupyterLab ≥ 4.1 sí). El código de texto de arriba describe el mismo grafo.

## 7. Cómo viaja la decisión entre agentes

Los agentes no comparten variables globales. El workflow incrusta la decisión del Orquestador **dentro del mensaje** que recibe el siguiente agente, con un marcador de control. Esa preparación la hacen dos funciones puras (`_contexto_correlacion` y `_contexto_criticidad`) que podemos ejecutar aquí con mensajes inventados.

> Son funciones internas (empiezan por `_`): las usamos solo para enseñar.

In [ ]:
from agent_framework import Message

decision = m.DecisionOrquestador(
    ruta="analisis_avanzado", asset_id="IT-004", cve_id=None,
    accion_explicita="ninguna", respuesta_directa=None, motivo="Análisis completo")

historial = [
    Message(role="user", contents=["Analiza las vulnerabilidades y la prioridad de IT-004."]),
    Message(role="assistant", contents=[decision.model_dump_json()]),   # lo que "diría" el Orquestador
]
para_correlacion = m._contexto_correlacion(historial)
print("=== Lo que recibe AgenteCorrelacion ===")
print(para_correlacion[0].text)

In [ ]:
# Y lo que recibe AgenteCriticidad: la petición + la salida (inventada) de Correlación
conversacion = para_correlacion + [
    Message(role="assistant", contents=["(aquí iría el resumen de candidatas de AgenteCorrelacion)"])]
para_criticidad = m._contexto_criticidad(conversacion)
print("=== Lo que recibe AgenteCriticidad ===")
print(para_criticidad[0].text)

# Si la decisión no es "analisis_avanzado", Criticidad NO ejecuta nada:
solo_correlacion = m.DecisionOrquestador(
    ruta="correlacion", asset_id="IT-004", cve_id=None,
    accion_explicita="ninguna", respuesta_directa=None, motivo="Solo candidatas")
ctx_c = m._contexto_correlacion([historial[0], Message(role="assistant", contents=[solo_correlacion.model_dump_json()])])
bloqueado = m._contexto_criticidad(ctx_c + [Message(role="assistant", contents=["texto"])])
print("\n=== Con ruta 'correlacion' ===")
print(bloqueado[0].text)

## 8. Respuesta directa verificada

Si el Orquestador elige *respuesta directa* pero no aporta texto, el nodo final **no inventa nada**: lee el inventario con la herramienta de solo lectura. Y si el activo no existe, lo dice.

In [ ]:
print(m._respuesta_inventario_verificada("IT-004"))
print()
print(m._respuesta_inventario_verificada("IT-999"))

## 9. Comprobación de integridad

Volvemos a calcular la huella SHA-256 de cada archivo de `datos/` y la comparamos con la del principio.

In [ ]:
huellas_despues = u.huellas_datos()
ok = u.informe_integridad(huellas_antes, huellas_despues)
assert ok, "¡Algún dato ha cambiado! Restaura el repositorio con git."

## Resumen y siguientes pasos

- Las **herramientas** son Python determinista; el **LLM solo decide cuál usar** y redacta.
- El **contrato** `DecisionOrquestador` hace que el enrutamiento no dependa de interpretar texto.
- El **grafo** tiene cuatro nodos y las rutas se eligen con condiciones sobre la decisión estructurada.
- Todo se ejecutó **sin red y sin modificar datos** ✔.

**Ejercicios (cambia el identificador y vuelve a ejecutar las celdas del apartado 3):**
1. Consulta `IT-001` y `OT-001`. ¿Cambian los totales (A) y (B) del cribado? ¿Por qué?
2. Evalúa `IT-001` con `evaluar_activo`. ¿Qué CVE tienen regla específica?
3. En el apartado 7 cambia `accion_explicita` a `"actualizar_novedades"`: ¿qué instrucción recibe `AgenteCorrelacion`? (Recuerda: en este paquete esa herramienta ni siquiera está registrada.)

➡️ Siguiente: **`entrepinares_02_workflow_foundry.ipynb`**, donde el workflow se ejecuta de verdad contra un modelo de Foundry.